# Lab 00b: Security Primer for the Labs

*AI Cybersecurity Masterclass · Lab 00b · AI60secLab*

**Research question:** What does a UNSW-NB15 record actually describe on a network, what do its attack categories mean, and what can this data never tell a defender?

Every later lab trains models on these records. This primer makes sure you know what the numbers are before you trust anything a model says about them.

**Learning objectives**
- explain what a network flow record is and what it leaves out;
- read the main feature families (basic, content, timing, connection counts);
- describe each of the nine attack categories in plain security terms;
- spot three properties of this dataset that would mislead a naive model;
- separate what flow data can *suggest* from what it can *prove*.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_unsw, load_feature_dictionary

train, test = load_unsw(DATA_DIR)
feature_dictionary = load_feature_dictionary(DATA_DIR)
feature_dictionary.columns = [c.strip() for c in feature_dictionary.columns]
feature_dictionary["name_key"] = feature_dictionary["Name"].str.lower().str.replace(" ", "")

print("Training rows:", len(train), "| Testing rows:", len(test))

## 1. One row is one conversation summary, not a packet

A **flow** summarizes one network conversation between two endpoints: how long it lasted, how many packets and bytes went each way, what protocol and service it used, and how the TCP connection ended.

The original UNSW-NB15 capture had 49 fields. The prepared train/test partition used in these labs keeps the behavioral fields and **drops the identifying ones**. Check which fields are missing.

In [ ]:
present = set(train.columns.str.lower())
renamed_in_partition = {
    "smeansz": "smean", "dmeansz": "dmean", "res_bdy_len": "response_body_len",
    "sintpkt": "sinpkt", "dintpkt": "dinpkt",
}
feature_dictionary["in_prepared_partition"] = (
    feature_dictionary["name_key"].replace(renamed_in_partition).isin(present)
)

dropped = feature_dictionary.loc[~feature_dictionary["in_prepared_partition"], ["Name", "Description"]]
display(dropped)

No source or destination IP, no ports, no timestamps. That means:
- no **identity**: you cannot say which host or user did anything;
- no **timeline**: you cannot order events or train on the past and test on the future (Lab 07 returns to this);
- no **payload**: you never see the content of the traffic, only its shape.

## 2. How a TCP connection ends: the `state` field

TCP opens a connection with a handshake (SYN, SYN-ACK, ACK) and closes it with FIN packets or an abrupt RST. The `state` field records how the conversation looked when the flow was summarized. The most common values here:

| State | Plain meaning |
|---|---|
| FIN | the connection closed normally |
| CON | the connection was still established |
| INT | an interrupted or one-sided exchange, typical of scans and floods that never complete a conversation |
| REQ | a request with no completed reply |
| RST | the connection was reset abruptly |

In [ ]:
state_by_label = pd.crosstab(train["state"], train["attack_cat"])
display(state_by_label.loc[["FIN", "INT", "CON", "REQ", "RST"]])

## 3. Feature families

| Family | Example fields | What it tells you |
|---|---|---|
| Basic | `proto`, `service`, `state`, `dur`, `spkts`, `dpkts`, `sbytes`, `dbytes` | size, direction, and length of the conversation |
| TTL and loss | `sttl`, `dttl`, `sloss`, `dloss` | how many hops packets survived, retransmissions |
| Timing | `sinpkt`, `dinpkt`, `sjit`, `djit`, `tcprtt`, `synack`, `ackdat` | spacing between packets and handshake speed |
| Content | `trans_depth`, `response_body_len`, `ct_flw_http_mthd`, `is_ftp_login` | light application-level hints for HTTP and FTP |
| Connection counts | `ct_srv_src`, `ct_dst_ltm`, `ct_src_dport_ltm`, ... | how many recent connections share the same service, source, or destination |

The `ct_*` counts look at the last 100 connections. They are the closest thing in this data to context: one flow alone looks innocent, but the same source hitting the same port a hundred times does not.

## 4. The nine attack categories

| Category | What it is | Closest ATT&CK lens (a hypothesis, not a mapping) |
|---|---|---|
| Fuzzers | sending malformed or random input to make a program or service crash or misbehave | reconnaissance (vulnerability discovery) |
| Analysis | port scans, spam, and HTML-file penetration attempts | reconnaissance, discovery |
| Backdoor | bypassing normal authentication to keep covert remote access | persistence, command-and-control |
| DoS | exhausting a service or network so legitimate users cannot reach it | impact |
| Exploits | abusing a known vulnerability to make a system do something unintended | initial-access, execution |
| Generic | attacks that work against a class of systems regardless of their specific design (in UNSW-NB15, mostly cipher-related traffic over DNS/UDP) | no clean mapping |
| Reconnaissance | probing and scanning to gather information about targets | reconnaissance, discovery |
| Shellcode | small payload code delivered to take control of a process after exploitation | execution |
| Worms | self-replicating malware that spreads to other hosts on its own | lateral-movement |

Lab 18 shows how to work with ATT&CK data properly. Here the point is simpler: a flow label is a **category of traffic**, not a confirmed technique.

## 5. What each category looks like as a flow

Median values per category. Look for categories that are easy to tell apart, and categories that look the same.

In [ ]:
fingerprint_columns = ["dur", "spkts", "dpkts", "sbytes", "dbytes", "sttl", "ct_srv_src"]
fingerprint = train.groupby("attack_cat")[fingerprint_columns].median()
fingerprint["top_proto"] = train.groupby("attack_cat")["proto"].agg(lambda s: s.value_counts().index[0])
fingerprint["top_service"] = train.groupby("attack_cat")["service"].agg(lambda s: s.value_counts().index[0])
fingerprint["top_state"] = train.groupby("attack_cat")["state"].agg(lambda s: s.value_counts().index[0])
display(fingerprint.round(3))

## 6. Three properties that will mislead a naive model

### 6a. The class balance is upside down

In [ ]:
balance = pd.DataFrame({
    "train_attack_share": [train["label"].mean()],
    "test_attack_share": [test["label"].mean()],
})
display(balance.round(3))

Most rows are attacks. On a real network the overwhelming majority of flows are normal. Precision measured here will look far better than it would in production, where the same false-positive *rate* produces many more false alarms per real attack. Labs 06, 08 and 09 deal with this.

### 6b. One field does a lot of the work

In [ ]:
ttl_table = pd.crosstab(train["sttl"], train["label"])
ttl_table = ttl_table[ttl_table.sum(axis=1) > 1000]
display(ttl_table)

def ttl_rule_accuracy(frame):
    return ((frame["sttl"].isin([62, 254])).astype(int) == frame["label"]).mean()

comparison = pd.DataFrame({
    "split": ["train", "test"],
    "always_say_attack": [train["label"].mean(), test["label"].mean()],
    "one_line_ttl_rule": [ttl_rule_accuracy(train), ttl_rule_accuracy(test)],
})
display(comparison.round(3))

`sttl` is the time-to-live of packets from the source. A TTL of 31 appears only on normal traffic, and 254 is overwhelmingly attack traffic. In this testbed, normal and attack traffic largely came from different machines, so `sttl` mostly encodes *where the traffic was generated*, not what it did.

A one-line rule with no learning lifts accuracy well above the "always say attack" baseline, and it works noticeably better on train than on test: the artifact is not even stable between the two partitions. A model can score well by learning *the lab setup* instead of attack behavior. That is a **dataset artifact**, and it will not transfer to another network. Labs 07 and 16 come back to why benchmark scores can overstate real performance.

### 6c. Some categories are indistinguishable in flow data

In [ ]:
look_alikes = fingerprint.loc[["Analysis", "Backdoor", "DoS"], fingerprint_columns + ["top_proto", "top_state"]]
display(look_alikes)

subset = train[train["attack_cat"].isin(["Analysis", "Backdoor", "DoS"])]
identical_shape = subset.groupby("attack_cat").apply(
    lambda g: ((g["spkts"] == 2) & (g["dpkts"] == 0) & (g["sbytes"] == 200)).mean(),
    include_groups=False,
)
print("Share of rows with exactly 2 packets out, 0 back, 200 bytes:")
display(identical_shape.round(3).rename("share").to_frame())

Large parts of three different attack categories have the same flow shape. No model can separate rows that are identical in every feature it sees. When Lab 03 shows poor recall for these classes, this is the reason: the information is not in the data.

## 7. What flow data can suggest, and what it cannot prove

| Flow data can suggest | Flow data cannot prove |
|---|---|
| scan-like or flood-like behavior | who the attacker is |
| an unusual volume or direction of traffic | intent |
| a connection pattern consistent with a known category | the exact ATT&CK technique used |
| which flows deserve an analyst's attention | that an attack succeeded |

Proving those needs endpoint, identity, application, or packet-level evidence.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
train["attack_cat"].value_counts().sort_values().plot(kind="barh", ax=ax)
ax.set_xlabel("Training rows")
ax.set_title("Rows per category: Worms and Shellcode are rare")
plt.tight_layout()
plt.show()

## Your analysis

Answer in a Markdown cell:

1. Which three fields would you check first to tell a port scan from a normal web session, and why?
2. Why does dropping IP addresses and timestamps make some validation designs impossible?
3. If a model's top feature is `sttl`, what should you conclude about its likely performance on a different network?
4. Analysis, Backdoor, and DoS share a flow shape. What additional telemetry would you need to tell them apart?
5. Write one sentence an analyst could put in a ticket for a flow the model labels "Reconnaissance", without overclaiming.

**Takeaway:** a model only knows what its features contain. Know the features, the artifacts, and the blind spots before you trust a score.